In [0]:
-- Creating a Gold layer for BI Consumption

SELECT date_format(formated_pickup_date,"yyyy-MM") as trip_year,pickup_service_zone,dropoff_service_zone,pickup_borough,dropoff_borough,round(sum(fare_amount),2) as total_fare_amount, round(sum(tip_amount),2) as total_tip_amount, round(sum(total_amount),2) as total_revenue, count(*) as total_trips
FROM nycyellow.silver.cleaned_trip_data_with_zones
where year(formated_pickup_date) = 2026 AND pickup_borough NOT IN ('Unknown','N/A') AND dropoff_service_zone != 'N/A'
GROUP BY  trip_year,pickup_borough,pickup_service_zone,dropoff_service_zone,dropoff_borough
ORDER BY trip_year DESC


In [0]:
SELECT payment_status, avg(try_divide(tip_amount, fare_amount)*100) as avg_tip_amount
FROM nycyellow.silver.cleaned_trip_data_with_zones
GROUP BY payment_status

In [0]:
SELECT dayofweek(formated_pickup_date) as pickup_day_of_week, dayname(formated_pickup_date) as pickup_day_name, month(formated_pickup_date) as pickup_month, pickup_borough, count(*) as count
FROM nycyellow.silver.cleaned_trip_data_with_zones
WHERE pickup_borough NOT IN ('Unknown','N/A')
GROUP BY pickup_day_of_week,pickup_day_name,pickup_month, pickup_borough
ORDER BY pickup_month ASC, pickup_day_of_week ASC

In [0]:
SELECT DISTINCT dropoff_service_zone, dropoff_borough
FROM nycyellow.silver.cleaned_trip_data_with_zones 
WHERE dropoff_service_zone = 'N/A' OR dropoff_borough = 'N/A'

In [0]:
-- Final Query 

CREATE OR REPLACE TABLE nycyellow.gold.trip_summary AS (
WITH base AS (
    SELECT
        date_format(formated_pickup_date, 'yyyy-MM') AS trip_year_month,
        month(formated_pickup_date)                  AS pickup_month,
        dayofweek(formated_pickup_date)               AS pickup_day_of_week,
        dayname(formated_pickup_date)                 AS pickup_day_name,
        hour(formated_pickup_date)                    AS pickup_hour,
        pickup_borough,
        pickup_service_zone,
        dropoff_borough,
        dropoff_service_zone,
        payment_status,
        CASE WHEN pickup_service_zone IN ('Airports', 'EWR')
               OR dropoff_service_zone IN ('Airports', 'EWR')
             THEN true ELSE false END                 AS is_airport_trip,
        fare_amount,
        tip_amount,
        total_amount
    FROM nycyellow.silver.cleaned_trip_data_with_zones
    WHERE pickup_borough NOT IN ('Unknown', 'N/A')
        AND dropoff_borough NOT IN ('Unknown', 'N/A')
        AND pickup_service_zone NOT IN ('Unknown', 'N/A')
        AND dropoff_service_zone NOT IN ('Unknown', 'N/A')
)
SELECT
    trip_year_month,
    pickup_month,
    pickup_day_of_week,
    pickup_day_name,
    pickup_hour,
    pickup_borough,
    pickup_service_zone,
    dropoff_borough,
    dropoff_service_zone,
    payment_status,
    is_airport_trip,
    round(sum(fare_amount), 2)                                        AS total_fare_amount,
    round(sum(tip_amount), 2)                                         AS total_tip_amount,
    round(avg(CASE WHEN fare_amount > 0 THEN tip_amount / fare_amount * 100 END), 2) AS avg_tip_pct,
    round(sum(total_amount), 2)                                       AS total_revenue,
    count(*)                                                          AS total_trips
FROM base
GROUP BY trip_year_month, pickup_month, pickup_day_of_week, pickup_day_name, pickup_hour,
         pickup_borough, pickup_service_zone, dropoff_borough, dropoff_service_zone,
         payment_status, is_airport_trip
ORDER BY trip_year_month DESC
)

In [0]:
CREATE OR REPLACE VIEW nycyellow.gold.month_over_month_summary AS (
    SELECT
    trip_year_month,
    sum(total_trips)   AS monthly_trips,
    sum(total_revenue) AS monthly_revenue,
    round(
      (sum(total_revenue) - lag(sum(total_revenue)) OVER (ORDER BY trip_year_month))
      / lag(sum(total_revenue)) OVER (ORDER BY trip_year_month) * 100, 2
    ) AS revenue_growth_pct
FROM nycyellow.gold.trip_summary
GROUP BY trip_year_month
ORDER BY trip_year_month
)